# TTTR Data Processing and Intensity Image Export Pipeline

## Overview
This script processes time-tagged time-resolved (TTTR) data from fluorescence microscopy experiments, extracting intensity images from different detection channels and time windows. It's designed for FRET (Förster Resonance Energy Transfer) experiments using PIE (Pulsed Interleaved Excitation) detection.

## Input/Output
- **Input**: `*.ptu` files (TTTR data from PicoQuant systems)
- **Output**: 
  - Individual channel time series: `*_green_s.tif`, `*_green_p.tif`, `*_red_s_prompt.tif`, etc.
  - Summary images: `*_all_SUM.tif` (total intensity)
  - Overview plots: `*_overview.png` (4-panel visualization)
  
## Experimental Setup
- **PIE Detection**: 40 MHz pulsed interleaved excitation
- **Polarization Channels**: s-polarized and p-polarized detection for each color
- **Time Windows**: 
  - Prompt (0-2500 ch): Direct excitation and FRET-sensitized emission
  - Delay (2500-5000 ch): Direct acceptor excitation
  
## Configuration Parameters
- **Channel mapping**: Green (0,3), Red (1,2) for s/p polarizations
- **Time binning**: 10 ps bins
- **Time windows**: Prompt (0-25 ns), Delay (25-50 ns)
- **Output format**: 16-bit TIFF files

## Processing Steps
1. Load TTTR data from .ptu files
2. Extract intensities for each channel/time window combination
3. Generate summary images and visualizations
4. Export time series and integrated images
5. Create overview plots for quality control

In [5]:
# Import required libraries
import tttrlib                           # TTTR data processing
import pylab as plt                      # Matplotlib plotting interface
from skimage import io                   # Image I/O operations
import numpy as np                       # Numerical operations
from skimage.util import img_as_uint     # Image format conversion
from mpl_toolkits.axes_grid1 import make_axes_locatable   # Subplot formatting
import glob                              # File pattern matching
import os                                # Operating system interface

## CONFIGURATION PARAMETERS

In [6]:
# Define input file pattern for batch processing
path = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_Files/Cells/*/*.ptu'

In [7]:
# Channel assignments for different detection paths
# These correspond to the physical detector channels in the setup
green_s_ch = 0     # Green perpendicular-polarized detection channel
green_p_ch = 3     # Green parallel-polarized detection channel
red_s_ch = 1       # Red perpendicular-polarized detection channel
red_p_ch = 2       # Red parallel-polarized detection channel

# Define time windows for PIE detection (in channel numbers)
# PIE uses pulsed excitation with different delay times
prompt_range = 0, 2500      # Early time window: donor excitation & donor / acceptor emission
delay_range = 2500, 5000    # Late time window: direct acceptor excitation & acceptor emission

## BATCH PROCESSING OF TTTR FILES

In [8]:
# Process each .ptu file in the specified directory structure
for file in glob.glob(path):
    # Extract base filename without extension for output naming
    filename = os.path.abspath(file).split(".")[0]
    
    # Load TTTR data and initialize CLSM image container
    data = tttrlib.TTTR(file)
    clsm_image = tttrlib.CLSMImage(tttr_data=data)

    print('Processing....' + filename)
    
    # =========================================================================
    # GREEN CHANNEL PROCESSING (Donor)
    # =========================================================================
    
    # Extract green s-polarized intensities in prompt time window
    # This captures donor emission and FRET-quenched donor signal
    clsm_image.fill(data, channels=[green_s_ch], micro_time_ranges=[prompt_range])
    int_green_s = clsm_image.intensity
    
    # Extract green p-polarized intensities in prompt time window
    clsm_image.fill(data, channels=[green_p_ch], micro_time_ranges=[prompt_range])
    int_green_p = clsm_image.intensity
    
    # Sum both polarizations and integrate over time for total green signal
    SUM_green = int_green_s.sum(axis=0) + int_green_p.sum(axis=0)
    
    # =========================================================================
    # RED PROMPT PROCESSING (FRET-sensitized acceptor)
    # =========================================================================
    
    # Extract red channel during prompt excitation time window
    # This captures FRET-sensitized acceptor emission
    clsm_image.fill(data, channels=[red_s_ch], micro_time_ranges=[prompt_range])
    int_red_s_prompt = clsm_image.intensity
    clsm_image.fill(data, channels=[red_p_ch], micro_time_ranges=[prompt_range])
    int_red_p_prompt = clsm_image.intensity
    
    # Sum and integrate red prompt signal
    SUM_red_prompt = int_red_s_prompt.sum(axis=0) +int_red_p_prompt.sum(axis=0)
    
    # =========================================================================
    # RED DELAY PROCESSING (Direct acceptor excitation)
    # =========================================================================
    
    # Extract red channel during delay excitation time window
    # This captures direct acceptor excitation (reference signal)
    clsm_image.fill(data, channels=[red_s_ch], micro_time_ranges=[delay_range])
    int_red_s_delay = clsm_image.intensity
    clsm_image.fill(data, channels=[red_p_ch], micro_time_ranges=[delay_range])
    int_red_p_delay = clsm_image.intensity
    
    # Sum and integrate red delay signal 
    SUM_red_delay = int_red_s_delay.sum(axis=0) + int_red_p_delay.sum(axis=0)
    
    # =========================================================================
    # TOTAL SIGNAL CALCULATION
    # =========================================================================
    
    # Combine all channels and time windows for total intensity
    # This image is used for segmentation & cell ROI definition
    SUM_all = SUM_green + SUM_red_prompt + SUM_red_delay
    
    # =========================================================================
    # VISUALIZATION AND QUALITY CONTROL
    # =========================================================================
    
    # Create 4-panel overview plot for quality assessment
    
    # Panel 1: Green channel (donor)    
    ax = plt.subplot(221)
    im = ax.imshow(SUM_green, cmap='plasma')
    ax.set_title('green Intensity')
    ax.axis('off')
    # Add colorbar with proper spacing
    divider = make_axes_locatable(ax)
    cax0 = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax0)
    
    # Panel 2: Red prompt (FRET-sensitized)
    ax = plt.subplot(222)
    im = ax.imshow(SUM_red_prompt, cmap='plasma')
    ax.set_title('red prompt Intensity')
    ax.axis('off')
    divider = make_axes_locatable(ax)
    cax1 = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax1)
    
    # Panel 3: Red delay (direct acceptor)
    ax = plt.subplot(223)
    im = ax.imshow(SUM_red_delay, cmap='plasma')
    ax.set_title('red delay Intensity')
    ax.axis('off')
    divider = make_axes_locatable(ax)
    cax2 = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax2)
    
    # Panel 4: Total intensity
    ax = plt.subplot(224)
    im = ax.imshow(SUM_all, cmap='plasma')
    ax.set_title('Sum Intensity')
    ax.axis('off')
    divider = make_axes_locatable(ax)
    cax3 = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax3)
    
    # Format and save overview plot
    plt.tight_layout()
    plt.savefig(filename + '_overview.png')
    plt.close()  # Close figure to free memory

    # =========================================================================
    # DATA EXPORT
    # =========================================================================
    
    # Export individual polarization time series as 16-bit TIFF files
    # These preserve the temporal information for subsequent analysis
    
    # Green channel time series (donor)
    io.imsave(filename + '_green_s.tif', img_as_uint(int_green_s), check_contrast=False)
    io.imsave(filename + '_green_p.tif', img_as_uint(int_green_p), check_contrast=False)
    
    # Red prompt time series (FRET-sensitized acceptor)
    io.imsave(filename + '_red_s_prompt.tif', img_as_uint(int_red_s_prompt), check_contrast=False)
    io.imsave(filename + '_red_p_prompt.tif', img_as_uint(int_red_p_prompt), check_contrast=False)
    
    # Red delay time series (direct acceptor excitation)
    io.imsave(filename + '_red_s_delay.tif', img_as_uint(int_red_s_delay), check_contrast=False)
    io.imsave(filename + '_red_p_delay.tif', img_as_uint(int_red_p_delay), check_contrast=False)
    
    # Export integrated intensity image (used for segmentation)
    io.imsave(filename + '_all_SUM.tif', img_as_uint(SUM_all), check_contrast=False)


Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\cytosolic\eGFP_44


C:\mambaforge\envs\FLIM\Lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 226 fits in uint16
  return _convert(image, np.uint16, force_copy)


Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\DA_1-5\low_FRET_1-5_64


C:\mambaforge\envs\FLIM\Lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 508 fits in uint16
  return _convert(image, np.uint16, force_copy)


Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\DA_1-5\low_FRET_1-5_66


...

C:\mambaforge\envs\FLIM\Lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 74 fits in uint16
  return _convert(image, np.uint16, force_copy)


Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\Donly\high_Donly_38


C:\mambaforge\envs\FLIM\Lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 177 fits in uint16
  return _convert(image, np.uint16, force_copy)


Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_Files\Cells\Donly\high_Donly_40


C:\mambaforge\envs\FLIM\Lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 172 fits in uint16
  return _convert(image, np.uint16, force_copy)


 
## OUTPUT FILE SUMMARY

Generated files for each input .ptu file:

TIME SERIES DATA (for temporal analysis):
- *_green_s.tif       : Green s-polarized time series (donor)
- *_green_p.tif       : Green p-polarized time series (donor)  
- *_red_s_prompt.tif  : Red s-polarized prompt time series (FRET)
- *_red_p_prompt.tif  : Red p-polarized prompt time series (FRET)
- *_red_s_delay.tif   : Red s-polarized delay time series (acceptor)
- *_red_p_delay.tif   : Red p-polarized delay time series (acceptor)

INTEGRATED IMAGES:
- *_all_SUM.tif       : Total intensity sum (used for cell segmentation)

QUALITY CONTROL:
- *_overview.png      : 4-panel visualization for data inspection

These files serve as input for:
- Cell segmentation (Script 3)
- Number & Brightness analysis (Script 5) 
- Multi-region segmentation (Scripts 6a-c)
- FRET efficiency calculations